In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import utils.get_ahn_tiles as get_ahn_tiles
import utils.las_utils as las_utils

import nest_asyncio
nest_asyncio.apply()

import geopandas as gpd
from config import TILE_DIR, AHN_RAW_DIR, AHN_GRID_SHP, SETUP_TILECODES

In [6]:
tilecodes = SETUP_TILECODES

data_dir      = TILE_DIR
ahn_grid_path = AHN_GRID_SHP
output_dir    = AHN_RAW_DIR

crs = "EPSG:28992"

In [7]:
gdf_ahn = gpd.read_file(ahn_grid_path)
ahn_tiles = []

for tilecode in tilecodes:
    # Search recursively — LAZ files may be nested inside subfolders
    matches = list(Path(data_dir).rglob(f"{tilecode}.laz")) + \
              list(Path(data_dir).rglob(f"{tilecode}.LAZ"))
    if not matches:
        print(f"  [skip] {tilecode} — LAZ file not found under {data_dir}")
        continue
    input_file = matches[0]
    print(f"  Found: {input_file}")

    poly = las_utils.build_convex_hull_polygon(input_file)
    centroid = poly.centroid

    gdf_point = gpd.GeoDataFrame(
        {"tilecode": [tilecode]},
        geometry=[centroid],
        crs=crs
    )

    joined = gpd.sjoin(gdf_point, gdf_ahn)
    ahn_tiles.extend(joined["GT_AHNSUB"].unique().tolist())

ahn_tiles = list(set(ahn_tiles))

  Found: data/input/pointcloud/raw/120300_489300.laz
  Found: data/input/pointcloud/raw/120300_488900.laz


In [8]:
await get_ahn_tiles.download_all_tiles(
    ahn_tiles,
    output_dir,
    "https://geotiles.citg.tudelft.nl/AHN5_T/{code}.LAZ"
)

100%|██████████| 1/1 [00:00<00:00, 286.71it/s]

Skipping 25EZ1_16 (already exists)


In [ ]:
!pip install -r requirements.txt